In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.utils import shuffle
from pathlib import Path

# --- CONFIG ---
train_csv = Path("/path/to/training/data.csv")  # Path to training CSV
input_shape = (128, 128, 1)  # Adjust based on your spectrogram dimensions
batch_size = 32
epochs = 50

# --- Load Training Data ---
train_df = pd.read_csv(train_csv)

# --- Data Generator ---
class SpectrogramDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataframe, batch_size, input_shape, shuffle=True):
        self.dataframe = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.input_shape = input_shape
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.dataframe) / self.batch_size))

    def __getitem__(self, index):
        batch_df = self.dataframe.iloc[index * self.batch_size:(index + 1) * self.batch_size]
        X = np.zeros((len(batch_df), *self.input_shape), dtype=np.float32)
        y = np.zeros((len(batch_df),), dtype=np.float32)

        for i, (_, row) in enumerate(batch_df.iterrows()):
            try:
                arr = np.load(row["filepath"])
                arr = arr.astype(np.float32)

                # Ensure shape and channel dimension
                arr = tf.image.resize(arr[..., np.newaxis], self.input_shape[:2]).numpy()
                X[i] = arr
                y[i] = row["label"]
            except Exception as e:
                print(f"⚠️ Error loading {row['filepath']}: {e}")

        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            self.dataframe = shuffle(self.dataframe)

# --- Build VGG16-like Regressor ---
def build_vgg16_regressor(input_shape):
    model = models.Sequential()

    # Block 1
    model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same', input_shape=input_shape))
    model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))

    # Block 2
    model.add(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))

    # Block 3
    model.add(layers.Conv2D(256, (3, 3), activation='relu', padding='same'))
    model.add(layers.Conv2D(256, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))

    # Fully connected
    model.add(layers.Flatten())
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation='linear'))  # Regression output

    return model

# --- Create Training Generator ---
train_gen = SpectrogramDataGenerator(train_df, batch_size, input_shape)

# --- Compile Model ---
model = build_vgg16_regressor(input_shape)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# --- Callbacks ---
checkpoint = callbacks.ModelCheckpoint("vgg16_regression_model_trained_only_on_train_data.keras", save_best_only=False)

# --- Train Only on Training Data ---
model.fit(
    train_gen,
    epochs=epochs,
    callbacks=[checkpoint],
    verbose=1
)

print("\n✅ Model training complete.")


In [ ]:
from tensorflow.keras.models import load_model
from pathlib import Path
import contextlib

model_path = Path("vgg16_regression_model_trained_only_on_train_data.keras")
summary_txt = Path("model_summary.txt")

# Load model without compiling
model = load_model(model_path, compile=False)

# Print and save summary
print("\n📊 Model Summary:\n")
model.summary()

with open(summary_txt, "w") as f:
    with contextlib.redirect_stdout(f):
        model.summary()

print(f"\n📄 Model summary saved to: {summary_txt}")


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from pathlib import Path

# --- CONFIG ---
model_path = Path("vgg16_regression_model_trained_only_on_train_data.keras")
val_csv = Path("/path/to/validation/data.csv")
input_shape = (128, 128, 1)  # Update if your spectrograms have different shape

# --- Load model (no compile needed just for prediction) ---
model = tf.keras.models.load_model(model_path, compile=False)

# --- Load validation data ---
val_df = pd.read_csv(val_csv)

# --- Preprocess all validation data ---
X_val = np.zeros((len(val_df), *input_shape), dtype=np.float32)
y_true = np.zeros((len(val_df),), dtype=np.float32)

for i, (_, row) in enumerate(val_df.iterrows()):
    arr = np.load(row["filepath"]).astype(np.float32)

    # Add channel dimension and resize if needed
    arr = tf.image.resize(arr[..., np.newaxis], input_shape[:2]).numpy()

    X_val[i] = arr
    y_true[i] = row["label"]

# --- Predict with the model ---
y_pred = model.predict(X_val, batch_size=32)

# --- Compute Squared Error and MSE ---
squared_errors = (y_pred.flatten() - y_true) ** 2
mse = np.mean(squared_errors)

print("\n✅ Squared error for each validation sample:")
print(squared_errors)

print(f"\n📊 Mean Squared Error (Validation): {mse:.6f}")


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from pathlib import Path

# --- CONFIG ---
model_path = Path("vgg16_regression_model_trained_only_on_train_data.keras")
val_csv = Path("/path/to/training/data.csv")
input_shape = (128, 128, 1)  # Update if your spectrograms have different shape

# --- Load model (no compile needed just for prediction) ---
model = tf.keras.models.load_model(model_path, compile=False)

# --- Load validation data ---
val_df = pd.read_csv(val_csv)

# --- Preprocess all validation data ---
X_val = np.zeros((len(val_df), *input_shape), dtype=np.float32)
y_true = np.zeros((len(val_df),), dtype=np.float32)

for i, (_, row) in enumerate(val_df.iterrows()):
    arr = np.load(row["filepath"]).astype(np.float32)

    # Add channel dimension and resize if needed
    arr = tf.image.resize(arr[..., np.newaxis], input_shape[:2]).numpy()

    X_val[i] = arr
    y_true[i] = row["label"]

# --- Predict with the model ---
y_pred = model.predict(X_val, batch_size=32)

# --- Compute Squared Error and MSE ---
squared_errors = (y_pred.flatten() - y_true) ** 2
mse = np.mean(squared_errors)

print("\n✅ Squared error for each validation sample:")
print(squared_errors)

print(f"\n📊 Mean Squared Error (Validation): {mse:.6f}")


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from pathlib import Path

# --- CONFIGURATION ---
model_path = Path("vgg16_regression_model_trained_only_on_train_data.keras")
test_csv = Path("/path/to/testing/data.csv")
input_shape = (128, 128, 1)  # Modify if your spectrogram shape is different

# --- Load Model (no need to compile for prediction) ---
model = tf.keras.models.load_model(model_path, compile=False)

# --- Load Test Data ---
test_df = pd.read_csv(test_csv)

# --- Preprocess Test Data ---
X_test = np.zeros((len(test_df), *input_shape), dtype=np.float32)
y_true = np.zeros((len(test_df),), dtype=np.float32)

for i, (_, row) in enumerate(test_df.iterrows()):
    arr = np.load(row["filepath"]).astype(np.float32)

    # Ensure correct shape and resize if necessary
    arr = tf.image.resize(arr[..., np.newaxis], input_shape[:2]).numpy()
    X_test[i] = arr
    y_true[i] = row["label"]

# --- Predict with the Model ---
y_pred = model.predict(X_test, batch_size=32).flatten()

# --- Compute Root Mean Squared Error ---
mse = np.mean((y_pred - y_true) ** 2)
rmse = np.sqrt(mse)

print(f"\n📊 Root Mean Squared Error (Test): {rmse:.6f}")


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from pathlib import Path

# --- CONFIGURATION ---
model_path = Path("vgg16_regression_model_trained_only_on_train_data.keras")
test_csv = Path("/path/to/testing/data.csv")
input_shape = (128, 128, 1)  # Modify if your spectrogram shape is different

# --- Load Model (no need to compile for prediction) ---
model = tf.keras.models.load_model(model_path, compile=False)

# --- Load Test Data ---
test_df = pd.read_csv(test_csv)

# --- Preprocess Test Data ---
X_test = np.zeros((len(test_df), *input_shape), dtype=np.float32)
y_true = np.zeros((len(test_df),), dtype=np.float32)

for i, (_, row) in enumerate(test_df.iterrows()):
    arr = np.load(row["filepath"]).astype(np.float32)

    # Ensure correct shape and resize if necessary
    arr = tf.image.resize(arr[..., np.newaxis], input_shape[:2]).numpy()
    X_test[i] = arr
    y_true[i] = row["label"]

# --- Predict with the Model ---
y_pred = model.predict(X_test, batch_size=32).flatten()

# --- Compute RMSE ---
mse = np.mean((y_pred - y_true) ** 2)
rmse = np.sqrt(mse)

# --- Compute Coefficient of Determination (R²) ---
ss_res = np.sum((y_true - y_pred) ** 2)
ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
r2 = 1 - (ss_res / ss_tot)

# --- Print Results ---
print(f"\n📊 Root Mean Squared Error (Test): {rmse:.6f}")
print(f"📈 Coefficient of Determination (R²): {r2:.6f}")
